# Lab 04 · Enterprise Grounding with Azure AI Search

File Search (Lab 03) is great for a handful of documents. In production, technical
content lives in an **enterprise search index** shared across many apps, with
**vector + semantic hybrid** retrieval and governance.

Here we:
1. Build an **Azure AI Search** index (vector + semantic).
2. Chunk the manuals, generate **embeddings**, and upload them.
3. Attach the index to an agent with **`AzureAISearchTool`** and get cited answers.

> ⚠️ Synthetic training data — not affiliated with or endorsed by Schneider Electric.

## 1. Connect + resolve the Search connection

The Foundry project already has a **connection** to Azure AI Search. We fetch it
(with credentials) so we can create an index and upload documents. The connection
may authenticate by **API key** or **Entra ID (RBAC)** — `config.get_search_credential`
handles both.

In [1]:
import sys
from pathlib import Path

here = Path.cwd()
src = next((p / "src" for p in [here, *here.parents] if (p / "src" / "config.py").exists()), None)
if src and str(src) not in sys.path:
    sys.path.insert(0, str(src))

import config

from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents import SearchClient

project_client = config.get_project_client()
search_conn = config.get_search_connection(project_client)
search_cred = config.get_search_credential(search_conn)

INDEX_NAME = config.WORKSHOP_INDEX_NAME
index_client = SearchIndexClient(endpoint=search_conn.target, credential=search_cred)
print("Search endpoint :", search_conn.target)
print("Connection name :", search_conn.name)
print("Target index    :", INDEX_NAME)

Search endpoint : https://your-search-service.search.windows.net/
Connection name : your-foundry-aisearch-connection
Target index    : schneider-manuals-index


## 2. Create the index (vector + semantic)

`text-embedding-3-large` produces **3072-dim** vectors. We add an HNSW vector
profile for similarity search and a semantic configuration for re-ranking.

In [2]:
from azure.search.documents.indexes.models import (
    SearchIndex, SimpleField, SearchableField, SearchField, SearchFieldDataType,
    VectorSearch, HnswAlgorithmConfiguration, VectorSearchProfile,
    SemanticConfiguration, SemanticPrioritizedFields, SemanticField, SemanticSearch,
)

EMBED_DIMS = 3072  # text-embedding-3-large

fields = [
    SimpleField(name="id", type=SearchFieldDataType.String, key=True),
    SearchableField(name="product", type=SearchFieldDataType.String, filterable=True, facetable=True),
    SearchableField(name="title", type=SearchFieldDataType.String),
    SearchableField(name="content", type=SearchFieldDataType.String),
    SearchField(
        name="contentVector",
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        searchable=True, vector_search_dimensions=EMBED_DIMS,
        vector_search_profile_name="vp",
    ),
]
vector_search = VectorSearch(
    algorithms=[HnswAlgorithmConfiguration(name="hnsw")],
    profiles=[VectorSearchProfile(name="vp", algorithm_configuration_name="hnsw")],
)
semantic_search = SemanticSearch(configurations=[
    SemanticConfiguration(
        name="default",
        prioritized_fields=SemanticPrioritizedFields(
            title_field=SemanticField(field_name="title"),
            content_fields=[SemanticField(field_name="content")],
            keywords_fields=[SemanticField(field_name="product")],
        ),
    )
])
index_client.create_or_update_index(
    SearchIndex(name=INDEX_NAME, fields=fields, vector_search=vector_search, semantic_search=semantic_search)
)
print(f"✅ Index '{INDEX_NAME}' created/updated.")

✅ Index 'schneider-manuals-index' created/updated.


## 3. Chunk the manuals, embed, and upload

We split each manual into its `##` sections (a simple, effective chunking
strategy) and embed each chunk with `config.embed_texts`.

In [3]:
import re

docs = []
for path in sorted(config.MANUALS_DIR.glob("*.md")):
    text = path.read_text(encoding="utf-8")
    product = text.splitlines()[0].lstrip("# ").split(" — ")[0]
    for i, part in enumerate(re.split(r"\n(?=## )", text)):
        chunk = part.strip()
        if len(chunk) < 20:
            continue
        docs.append({
            "id": f"{path.stem}-{i}",
            "product": product,
            "title": chunk.splitlines()[0].lstrip("# ").strip(),
            "content": chunk,
        })

print(f"Prepared {len(docs)} chunks. Generating embeddings...")
vectors = config.embed_texts([d["content"] for d in docs])
for d, v in zip(docs, vectors):
    d["contentVector"] = v

SearchClient(endpoint=search_conn.target, index_name=INDEX_NAME, credential=search_cred).upload_documents(documents=docs)
print(f"✅ Uploaded {len(docs)} chunks to '{INDEX_NAME}'.")

Prepared 24 chunks. Generating embeddings...


✅ Uploaded 24 chunks to 'schneider-manuals-index'.


## 4. Create an agent that searches the index

`AzureAISearchTool` points the agent at our index via the project **connection**.
`query_type=SEMANTIC` enables semantic re-ranking; the agent returns answers with
inline citations like `【4:0†source】`.

In [4]:
from azure.ai.projects.models import (
    PromptAgentDefinition, AzureAISearchTool, AzureAISearchToolResource,
    AISearchIndexResource, AzureAISearchQueryType,
)

search_tool = AzureAISearchTool(
    azure_ai_search=AzureAISearchToolResource(
        indexes=[AISearchIndexResource(
            project_connection_id=search_conn.name,
            index_name=INDEX_NAME,
            query_type=AzureAISearchQueryType.SEMANTIC,
        )]
    )
)

search_agent = config.create_prompt_agent(
    project_client,
    name="technician-copilot-aisearch",
    instructions=config.TECH_PERSONA + """

You have an Azure AI Search tool over the product-manual knowledge base. Always
search it for fault codes, specs, safety, and procedures, and cite the sources it
returns. If nothing relevant is found, say so.""",
    tools=[search_tool],
)
print(f"✅ Search agent: {search_agent.name} (v{search_agent.version})")

✅ Search agent: technician-copilot-aisearch (v1)


## 5. Ask grounded questions

In [5]:
import time
time.sleep(3)  # give the index a moment to finish indexing

for q in [
    "What does fault code A140 mean on the PowerLogic PM8000 and what should I check?",
    "A Galaxy VS UPS shows E07. What is the first action?",
    "Which products mention IGBT stages, and why does that matter for safety?",
]:
    print("👤", q)
    print("🤖", config.ask(project_client, search_agent, q), "\n" + "-"*70)

👤 What does fault code A140 mean on the PowerLogic PM8000 and what should I check?


🤖 On the PM8000, fault/alarm code **A140 means “Voltage THD exceeds 8% alarm threshold.”**【4:0†source】

What to check first:
- **Confirm the alarm is real** by reviewing the meter’s **voltage THD** reading and event log.【4:0†source】【4:2†source】
- **Check the alarm setpoint** and compare it to the site’s power quality standard, since the manual specifically says to confirm the threshold setting.【4:0†source】
- **Look for nonlinear loads on the feeder**, especially **VFDs, UPS rectifiers, switch-mode power supplies, or large harmonic-producing loads**. The manual’s first action is to investigate nonlinear loads / VFDs on that feeder.【4:0†source】
- If the THD is unexpectedly high, verify the **voltage sensing/PT wiring and phase mapping** are correct before assuming a meter issue.

Safety:
- If you open panels or test live circuits, follow **LOTO where applicable**, wear **arc-flash PPE**, and use properly rated test equipment.  
- If you are checking associated VFDs or UPS equipment, reme

🤖 For a Galaxy VS showing **E07**, the **first action** is to **transfer the load to static bypass, isolate the UPS, then inspect the DC bus and rectifier IGBT stage before restart**【4:0†source】.

Safety first:
- Apply **LOTO** before servicing【4:1†source】.
- Use **arc-flash PPE** if the cabinet is energized【4:1†source】.
- The **DC capacitor bank can retain lethal charge for up to 5 minutes after isolation** — verify zero energy with a meter before touching anything【4:1†source】.

If you want, I can also give you a quick field troubleshooting sequence for **E07 / DC bus overvoltage**. 
----------------------------------------------------------------------
👤 Which products mention IGBT stages, and why does that matter for safety?


🤖 The manuals I found mention IGBT power stages in these products:

- **Galaxy VS UPS** — the overview states it uses **IGBT rectifier and inverter stages** in its power conversion path【4:1†source】.
- **Altivar Process ATV630 drive** — the overview states it uses an **IGBT inverter bridge** fed from a **DC bus** for PWM motor control【4:2†source】.

Why that matters for safety:

- **IGBT stages are tied to high-energy DC bus circuits.** In both UPSs and drives, the IGBT power section switches substantial DC voltage/current. That means a fault or service error can expose you to **lethal stored energy** and high fault energy.
- For the **ATV630**, the manual explicitly warns that the **DC bus retains lethal voltage after power-off**, requiring a **minimum 15-minute wait** and verification of **zero DC bus voltage** before touching terminals; **LOTO** and **arc-flash PPE** apply【4:3†source】.
- For the **Galaxy VS**, the UPS overview identifies a **480 V DC bus across the capacitor bank**【4:

## 🙌 Your turn

1. Switch the tool's `query_type` to `AzureAISearchQueryType.VECTOR_SEMANTIC_HYBRID`
   and re-create the agent. Compare the answers — hybrid blends keyword + vector.
2. Add a **filter** experiment: ask a question scoped to a single product and see
   whether the retrieved citations stay on-topic.

In [6]:
# 👉 Your experiment here.

## Clean up

We delete the agent. We keep the **index** if you want to reuse it in Lab 06;
uncomment the last line to delete it.

In [7]:
config.delete_agent(project_client, search_agent)
print("🗑️  Deleted search agent.")
# index_client.delete_index(INDEX_NAME); print("🗑️  Deleted index.")
print("Next: Lab 05 orchestrates multiple specialist agents.")

🗑️  Deleted search agent.
Next: Lab 05 orchestrates multiple specialist agents.
